# 📘 DATABASE MANAGEMENT AND DESIGN
### *A Practical Learning Series*

# Notebook 3.4 — SQL and Data Analysis — Advanced Level II
### Subqueries/Nested Queries, VIEW, Temp Tables, CTE, PARTITION BY + CTE

**By Amin Amirkhalili** — Business & Data Analyst

*Based on Chapter 6 (Sections 6-6 and 6-8) of:* **Database Management and Design** — Gove Allen, PhD · Gary Hansen, PhD · Robert Jackson, PhD


## 🗺️ Series Roadmap

This notebook is part of a practical learning series on database management and design. The series follows a logical progression — from understanding business needs, through conceptual modelling, all the way to writing SQL code on real databases.

| | |
|---|---|
| **1 — Fundamentals** | |
| Notebook 1 | Introduction: The Database Is the Heart of Information Systems |
| **2 — Database Management** | |
| Notebook 2.1 | Conceptual Data Model: From Conceptual Design to a Relational Model |
| Notebook 2.2 | Database Design: From Schema Creation to Data Management |
| **3 — SQL & Data Analysis** | |
| Notebook 3.1 | Beginner Level: SELECT, FROM, WHERE, ORDER BY |
| Notebook 3.2 | Intermediate Level: Functions, GROUP BY, HAVING |
| Notebook 3.3 | Advanced Level I: Joining Tables — Inner & Outer Joins |
| **Notebook 3.4** | Advanced Level II: Subqueries, Views, Temp Tables, CTEs **◀ You are here** |
| Notebook 3.5 | Data Cleaning with SQL *(coming soon)* |
| **4 — Data Mining** | |
| Notebook 4.1 | SQL in Data Mining *(coming soon)* |

## 📌 What This Notebook Covers

Subqueries in WHERE, SELECT and FROM (correlated and non-correlated), views, temp tables, CTEs, when to use each, and the PARTITION BY + CTE pattern.

## 💻 How to Use This Notebook

This is the **live practice version** — every query below runs for real.

1. Run the setup cell below **first** (▶ button on the left). It builds a small sample database with the same tables used in the series.
2. Run each query cell as you read. Change the queries — experiment!
3. To start fresh, just run the setup cell again.

> **Note on SQL dialects:** the reading copy of this notebook uses **SQL Server (SSMS)**. Here we run **SQLite** (built into Colab, no installation needed). The two are almost identical; wherever the syntax differs (e.g. `TOP` vs `LIMIT`, `+` vs `||` for text), a note shows both versions.

In [1]:
#@title ▶️ Run this cell first — it builds the practice database { display-mode: "form" }
# This cell creates a small sample database (SQLite) with the same tables used
# in the notebook: Customer, Manufacturer, Product, Sale, SaleItem, Employee,
# SalaryEmployee, WageEmployee, Purchase, Amin, SimplifiedSales, and more.
# It also defines q("...") which runs a SQL query and shows the result.

import sqlite3, math, datetime
import pandas as pd

conn = sqlite3.connect(":memory:")
conn.executescript("""
PRAGMA foreign_keys = ON;

CREATE TABLE Customer (
  CustomerID INT NOT NULL, FirstName VARCHAR(20), LastName VARCHAR(50),
  StreetAddress VARCHAR(80), City VARCHAR(40), State VARCHAR(2),
  PostalCode VARCHAR(10), Country VARCHAR(30), Phone VARCHAR(20),
  PRIMARY KEY (CustomerID));

CREATE TABLE Manufacturer (
  ManufacturerID INT NOT NULL, ManufacturerName VARCHAR(50),
  Address VARCHAR(80), City VARCHAR(40), State VARCHAR(2),
  PostalCode VARCHAR(10), Phone VARCHAR(20),
  PRIMARY KEY (ManufacturerID));

CREATE TABLE Product (
  ProductID INT NOT NULL, ProductName VARCHAR(60), ManufacturerID INT,
  Category VARCHAR(30), Color VARCHAR(20), Gender VARCHAR(1),
  ListPrice NUMERIC(8,2), Description VARCHAR(120),
  PRIMARY KEY (ProductID),
  FOREIGN KEY (ManufacturerID) REFERENCES Manufacturer(ManufacturerID));

CREATE TABLE Sale (
  SaleID INT NOT NULL, SaleDate DATE NOT NULL,
  Tax NUMERIC(8,2), Shipping NUMERIC(8,2), CustomerID INT,
  PRIMARY KEY (SaleID),
  FOREIGN KEY (CustomerID) REFERENCES Customer(CustomerID));

CREATE TABLE SaleItem (
  SaleID INT NOT NULL, ProductID INT NOT NULL, ItemSize NUMERIC(3,1) NOT NULL,
  Quantity INT NOT NULL, SalePrice NUMERIC(8,2) NOT NULL,
  PRIMARY KEY (SaleID, ProductID, ItemSize),
  FOREIGN KEY (SaleID) REFERENCES Sale(SaleID),
  FOREIGN KEY (ProductID) REFERENCES Product(ProductID));

CREATE TABLE Employee (
  EmployeeID INT NOT NULL, FirstName VARCHAR(20), LastName VARCHAR(50),
  Age INT, City VARCHAR(40), ManagerID INT,
  PRIMARY KEY (EmployeeID),
  FOREIGN KEY (ManagerID) REFERENCES Employee(EmployeeID));

CREATE TABLE SalaryEmployee (
  EmployeeID INT NOT NULL, Salary NUMERIC(10,2),
  PRIMARY KEY (EmployeeID),
  FOREIGN KEY (EmployeeID) REFERENCES Employee(EmployeeID));

CREATE TABLE WageEmployee (
  EmployeeID INT NOT NULL, Wage NUMERIC(8,2), MaxHours INT,
  PRIMARY KEY (EmployeeID),
  FOREIGN KEY (EmployeeID) REFERENCES Employee(EmployeeID));

CREATE TABLE Purchase (
  PurchaseID INT NOT NULL, PurchaseDate DATE, EmployeeID INT, ManufacturerID INT,
  PRIMARY KEY (PurchaseID),
  FOREIGN KEY (EmployeeID) REFERENCES Employee(EmployeeID),
  FOREIGN KEY (ManufacturerID) REFERENCES Manufacturer(ManufacturerID));

CREATE TABLE Amin (
  ID INT NOT NULL, FirstName VARCHAR(20), LastName VARCHAR(50),
  Age INT, Gender VARCHAR(10), PRIMARY KEY (ID));

CREATE TABLE Population (
  Country VARCHAR(40), City VARCHAR(40), Population INT);

CREATE TABLE CovidDeaths (
  location VARCHAR(40), continent VARCHAR(20), date DATE,
  total_cases INT, population INT);

INSERT INTO Manufacturer VALUES
 (1,'Nike','1 Bowerman Dr','Boston','NJ','07101','555-0101'),
 (2,'Adidas','5 Sport Ave','Boston','NJ','07102','555-0102'),
 (3,'Puma','9 Cat St','Chicago','IL','60601','555-0103'),
 (4,'Fila','3 Retro Rd','San Diego','CA','92101','555-0104'),
 (5,'Fashion Lab','7 Style Blvd','Los Angeles','CA','90001','555-0105'),
 (6,'GAP','11 Cotton Way','Concord','NH','03301','555-0106');

INSERT INTO Customer VALUES
 (1,'Sara','Lopez','12 Main St','Boston','VA','23220','USA','555-1001'),
 (2,'Amin','Amirkhalili','34 Oak Ave','Boston','MA','02108','USA','555-1002'),
 (3,'Kim','Nguyen','56 Pine Rd','Chicago','IL','60614','USA','555-1003'),
 (4,'John','Smith','78 Elm St','Richmond','VA','23230','USA','555-1004'),
 (5,'George','Brown','90 Cedar Ln','Springfield','IL','62701','USA','555-1005'),
 (6,'Kevin','Miller','21 Birch Ct','San Diego','CA','92103','USA','555-1006'),
 (7,'Sam','Wilson','43 Maple Dr','Portland','OR','97201','USA','555-1007'),
 (8,'Elli','Rahimi','65 Walnut St','Los Angeles','CA','90012','USA','555-1008');

INSERT INTO Product VALUES
 (1,'AirRun Sneaker',1,'Sneakers','Black','M',120,'Running sneaker'),
 (2,'AirRun Sneaker W',1,'Sneakers','White','F',115,'Running sneaker'),
 (3,'ClassicSandal',1,'Sandals','Brown','F',60,'Summer sandal'),
 (4,'StreetBoot',2,'Boots','Black','M',150,'Urban boot'),
 (5,'TrailBoot',2,'Boots','Brown','F',160,'Hiking boot'),
 (6,'SpeedCat Sneaker',3,'Sneakers','Red','M',95,'Classic sneaker'),
 (7,'BeachSandal',3,'Sandals','Gold','F',45,'Beach sandal'),
 (8,'RetroRunner',4,'Sneakers','Black','F',85,'Retro sneaker'),
 (9,'HeritageBoot',4,'Boots','Tan','M',140,'Heritage boot'),
 (10,'GlamHeel',5,'Heels','Black','F',130,'Evening heel'),
 (11,'CityFlat',5,'Flats','Blue','F',70,'Comfort flat'),
 (12,'CanvasClassic',6,'Sneakers','White','M',55,'Canvas sneaker'),
 (13,'WinterBoot',6,'Boots','Black','F',110,'Insulated boot');

INSERT INTO Sale VALUES
 (1001,'2026-07-03',8.40,5.00,1),
 (1002,'2026-07-02',6.20,5.00,2),
 (1003,'2026-06-28',12.10,0.00,3),
 (1004,'2026-06-15',4.75,5.00,4),
 (1005,'2026-05-30',9.90,7.50,2),
 (1006,'2026-01-18',7.25,5.00,5),
 (1007,'2026-01-05',3.60,0.00,6),
 (1008,'2025-12-20',10.10,5.00,1),
 (1009,'2025-12-08',5.45,5.00,8),
 (1010,'2026-07-10',6.80,0.00,3);

INSERT INTO SaleItem VALUES
 (1001,1,10.0,10,100.00),(1001,4,9.5,5,200.00),
 (1002,6,8.0,1,150.00),
 (1003,2,7.5,2,110.00),(1003,7,6.0,1,42.00),
 (1004,8,7.0,1,80.00),
 (1005,10,6.5,1,125.00),(1005,11,6.0,2,65.00),
 (1006,12,11.0,3,50.00),(1006,1,10.5,1,110.00),
 (1007,9,10.5,1,135.00),
 (1008,13,8.5,2,105.00),(1008,3,7.0,1,55.00),
 (1009,5,8.0,1,155.00),
 (1010,1,9.0,1,118.00);

INSERT INTO Employee VALUES
 (1,'Alan','Stone',55,'Boston',NULL),
 (2,'Ahmad','Karimi',41,'Boston',1),
 (3,'Elli','Moore',38,'Chicago',1),
 (4,'Luke','Perry',29,'San Diego',2),
 (5,'James','Chen',33,'San Diego',2),
 (6,'Alex','Diaz',26,'Portland',3);

INSERT INTO SalaryEmployee VALUES
 (1,95000),(2,72000),(3,68000),(4,80000);

INSERT INTO WageEmployee VALUES
 (5,28.50,40),(6,22.00,30);

INSERT INTO Purchase VALUES
 (501,'2025-12-05',2,1),
 (502,'2025-12-12',4,2),
 (503,'2025-12-27',5,3),
 (504,'2026-02-10',2,4),
 (505,'2026-03-22',3,5),
 (506,'2025-12-30',NULL,6);

INSERT INTO Amin VALUES
 (1,'Amin','Amirkhalili',30,'Male'),
 (2,'Amin','Amirkhalili',50,'Male'),
 (3,'Sara','Lopez',27,'Female'),
 (4,'Elli','Rahimi',35,'Female'),
 (5,'Sadra','Kazemi',22,'Male'),
 (6,'Hossein','Nouri',44,'Male'),
 (7,'Kim','Nguyen',31,'Female');

INSERT INTO Population VALUES
 ('Canada','Toronto',2794356),('Canada','Montreal',1762949),
 ('Canada','Vancouver',662248),('USA','New York',8804190),
 ('USA','Los Angeles',3898747),('USA','Chicago',2746388),
 ('Germany','Berlin',3644826),('Germany','Munich',1471508);

INSERT INTO CovidDeaths VALUES
 ('Canada','North America','2021-06-01',1385000,38000000),
 ('Canada','North America','2021-06-15',1401000,38000000),
 ('Canada','North America','2021-07-01',1412000,38000000),
 ('Germany','Europe','2021-06-01',3700000,83000000),
 ('Germany','Europe','2021-06-15',3717000,83000000),
 ('Germany','Europe','2021-07-01',3730000,83000000),
 ('Iran','Asia','2021-06-01',2950000,85000000),
 ('Iran','Asia','2021-06-15',3020000,85000000),
 ('Iran','Asia','2021-07-01',3230000,85000000);

CREATE TABLE SimplifiedSales AS
SELECT s.SaleID, s.SaleDate, s.CustomerID, p.ProductName, p.Category, p.Color,
       m.ManufacturerName, si.Quantity, si.SalePrice
FROM Sale s
JOIN SaleItem si ON s.SaleID = si.SaleID
JOIN Product p  ON si.ProductID = p.ProductID
JOIN Manufacturer m ON p.ManufacturerID = m.ManufacturerID;
""")

# --- Register SQL-Server-style functions so the book's code runs in SQLite ---
conn.create_function("LEN", 1, lambda s: len(s) if s is not None else None)
conn.create_function("CHARINDEX", 2, lambda sub, s: (s.find(sub) + 1) if (s is not None and sub is not None) else None)
conn.create_function("SQRT", 1, lambda x: math.sqrt(x) if x is not None else None)
conn.create_function("POWER", 2, lambda x, n: x ** n if x is not None else None)
conn.create_function("CEILING", 1, lambda x: math.ceil(x) if x is not None else None)
conn.create_function("FLOOR", 1, lambda x: math.floor(x) if x is not None else None)
conn.create_function("GETDATE", 0, lambda: datetime.date.today().isoformat())
conn.create_function("MONTH", 1, lambda d: int(str(d)[5:7]) if d else None)
conn.create_function("YEAR", 1, lambda d: int(str(d)[0:4]) if d else None)

class _StringAgg:
    def __init__(self): self.items = []
    def step(self, value, sep): self.sep = sep; self.items.append(str(value))
    def finalize(self): return self.sep.join(self.items) if self.items else None
conn.create_aggregate("STRING_AGG", 2, _StringAgg)

def q(sql):
    """Run a SQL query and show the result as a table."""
    sql_stripped = sql.strip().rstrip(";").strip()
    first = sql_stripped.split()[0].upper() if sql_stripped else ""
    if first in ("SELECT", "WITH"):
        return pd.read_sql_query(sql, conn)
    conn.executescript(sql)
    print("OK —", first, "executed.")

print("✅ Practice database is ready. Use q(\"...\") to run SQL. Example:")
q("SELECT * FROM Customer LIMIT 3")


✅ Practice database is ready. Use q("...") to run SQL. Example:


,CustomerID,FirstName,LastName,StreetAddress,City,State,PostalCode,Country,Phone
0,1,Sara,Lopez,12 Main St,Boston,VA,23220,USA,555-1001
1,2,Amin,Amirkhalili,34 Oak Ave,Boston,MA,02108,USA,555-1002
2,3,Kim,Nguyen,56 Pine Rd,Chicago,IL,60614,USA,555-1003


## Section 1 — Subqueries / Nested Queries

**Definition: subqueries are queries that come inside an outer/main query.**

- Subqueries can be **non-correlated** or **correlated** with the main query.
- Subqueries can come in the WHERE, SELECT, FROM, ... clauses.

### 1.1 Subqueries in the WHERE clause — non-correlated mode

Non-correlated subqueries are independent of the main queries that use them. Therefore, they run first. Then the main query runs.

**Query: what are the categories of products made by manufacturers located in New Jersey?**

In [2]:
q("""
SELECT DISTINCT Category
FROM Product
WHERE ManufacturerID IN
    (SELECT ManufacturerID
     FROM Manufacturer
     WHERE State = 'NJ')
""")

,Category
0,Sneakers
1,Sandals
2,Boots


Subqueries offer a partial alternative to the join:

In [3]:
q("""
SELECT DISTINCT P.Category
FROM Product P
JOIN Manufacturer M ON M.ManufacturerID = P.ManufacturerID
WHERE State = 'NJ'
""")

,Category
0,Sneakers
1,Sandals
2,Boots


**Another example — query: list sales of black sneakers.** With subqueries:

In [4]:
q("""
SELECT *
FROM Sale
WHERE SaleID IN
    (SELECT SaleID
     FROM SaleItem
     WHERE ProductID IN
         (SELECT ProductID
          FROM Product
          WHERE Category = 'Sneakers' AND Color = 'Black'))
""")

,SaleID,SaleDate,Tax,Shipping,CustomerID
0,1001,2026-07-03,8.40,5,1
1,1004,2026-06-15,4.75,5,4
2,1006,2026-01-18,7.25,5,5
3,1010,2026-07-10,6.80,0,3


### 1.2 Correlated subqueries

Correlated subqueries are subqueries whose value upon execution depends on the row being examined by the main query. In correlated subqueries, we usually need to make different copies of the main tables — one for the subquery and one for the main query.

**Query: list employees who receive a higher salary than their managers.**

In [5]:
q("""
SELECT FirstName, LastName FROM Employee E
JOIN SalaryEmployee SE ON E.EmployeeID = SE.EmployeeID
WHERE SE.Salary > (SELECT SM.Salary FROM Employee M
                   JOIN SalaryEmployee SM ON M.EmployeeID = SM.EmployeeID
                   WHERE M.EmployeeID = E.ManagerID)
""")

,FirstName,LastName
0,Luke,Perry


**Query: for each manufacturer, show its higher-priced products.** We define higher-priced products to be those whose list price is higher than the average list price for that manufacturer. Again, we need two copies of the only table we need:

In [6]:
q("""
SELECT ManufacturerID,
       ProductID,
       ProductName,
       ListPrice
FROM Product P1
WHERE ListPrice > (SELECT AVG(ListPrice)
                   FROM Product P2
                   WHERE P2.ManufacturerID = P1.ManufacturerID)
ORDER BY ManufacturerID
""")

,ManufacturerID,ProductID,ProductName,ListPrice
0,1,1,AirRun Sneaker,120
1,1,2,AirRun Sneaker W,115
2,2,5,TrailBoot,160
3,3,6,SpeedCat Sneaker,95
4,4,9,HeritageBoot,140
5,5,10,GlamHeel,130
6,6,13,WinterBoot,110


### 1.3 Subqueries in WHERE with aggregate functions

**Non-correlated — query: which employees receive a higher-than-average salary?**

In [7]:
q("""
SELECT FirstName, LastName, Salary
FROM Employee E JOIN SalaryEmployee SE ON E.EmployeeID = SE.EmployeeID
WHERE Salary >
    (SELECT AVG(Salary)
     FROM SalaryEmployee)
""")

,FirstName,LastName,Salary
0,Alan,Stone,95000
1,Luke,Perry,80000


**Correlated — query: which employees receive a salary higher than the average for employees reporting to the employee's manager?**

In [8]:
q("""
SELECT FirstName, LastName, Salary
FROM Employee E1 JOIN SalaryEmployee SE1 ON E1.EmployeeID = SE1.EmployeeID
WHERE Salary >
    (SELECT AVG(Salary)
     FROM Employee E2 JOIN SalaryEmployee SE2 ON E2.EmployeeID = SE2.EmployeeID
     WHERE E2.ManagerID = E1.ManagerID)
""")

,FirstName,LastName,Salary
0,Ahmad,Karimi,72000


### 1.4 IN vs EXISTS in the WHERE clause

IN and EXISTS are very similar, but:
- **IN**: it creates the list first, and then checks.
- **EXISTS**: it takes each row, and then checks.

In [9]:
q("""
SELECT *
FROM Customer
WHERE CustomerID IN
    (SELECT CustomerID
     FROM Sale)
""")

,CustomerID,FirstName,LastName,StreetAddress,City,State,PostalCode,Country,Phone
0,1,Sara,Lopez,12 Main St,Boston,VA,23220,USA,555-1001
1,2,Amin,Amirkhalili,34 Oak Ave,Boston,MA,02108,USA,555-1002
2,3,Kim,Nguyen,56 Pine Rd,Chicago,IL,60614,USA,555-1003
3,4,John,Smith,78 Elm St,Richmond,VA,23230,USA,555-1004
4,5,George,Brown,90 Cedar Ln,Springfield,IL,62701,USA,555-1005
5,6,Kevin,Miller,21 Birch Ct,San Diego,CA,92103,USA,555-1006
6,8,Elli,Rahimi,65 Walnut St,Los Angeles,CA,90012,USA,555-1008


In [10]:
q("""
SELECT *
FROM Customer C
WHERE EXISTS (
    SELECT 1
    FROM Sale S
    WHERE S.CustomerID = C.CustomerID
)
""")

,CustomerID,FirstName,LastName,StreetAddress,City,State,PostalCode,Country,Phone
0,1,Sara,Lopez,12 Main St,Boston,VA,23220,USA,555-1001
1,2,Amin,Amirkhalili,34 Oak Ave,Boston,MA,02108,USA,555-1002
2,3,Kim,Nguyen,56 Pine Rd,Chicago,IL,60614,USA,555-1003
3,4,John,Smith,78 Elm St,Richmond,VA,23230,USA,555-1004
4,5,George,Brown,90 Cedar Ln,Springfield,IL,62701,USA,555-1005
5,6,Kevin,Miller,21 Birch Ct,San Diego,CA,92103,USA,555-1006
6,8,Elli,Rahimi,65 Walnut St,Los Angeles,CA,90012,USA,555-1008


### 1.5 Subquery in the SELECT clause

- **Non-correlated**: `(SELECT ... FROM ...)` is a constant — the same value for every row.
- **Correlated**: depending on each row, the value is different: d1, d2, d3, ...

**Query: for each customer, the total value they have bought.**

In [11]:
q("""
SELECT CustomerID,
       FirstName,
       LastName,
       (SELECT SUM(SalePrice * Quantity)
        FROM SaleItem SI
        JOIN Sale S ON SI.SaleID = S.SaleID
        WHERE S.CustomerID = C.CustomerID) AS TotalSold
FROM Customer C
""")

,CustomerID,FirstName,LastName,TotalSold
0,1,Sara,Lopez,2265.0
1,2,Amin,Amirkhalili,405.0
2,3,Kim,Nguyen,380.0
3,4,John,Smith,80.0
4,5,George,Brown,260.0
5,6,Kevin,Miller,135.0
6,7,Sam,Wilson,NaN
7,8,Elli,Rahimi,155.0


### 1.6 Subquery in FROM

As a subquery in SELECT makes a **column**, a subquery in FROM makes a **table**. It creates a temporary table so we can select, join, or use WHERE on it. The big difference between a subquery and a VIEW (defined below) is that a subquery is gone after the query finishes, BUT the view is permanent.

**Query: show the total value of sales for each customer.**

In [12]:
q("""
SELECT C.CustomerID,
       FirstName,
       LastName,
       TotalSold
FROM Customer C
JOIN (SELECT CustomerID,
             SUM(SalePrice * Quantity) AS TotalSold
      FROM Sale S
      JOIN SaleItem SI
        ON S.SaleID = SI.SaleID
      GROUP BY CustomerID) SumQuery
  ON C.CustomerID = SumQuery.CustomerID
""")

,CustomerID,FirstName,LastName,TotalSold
0,1,Sara,Lopez,2265
1,2,Amin,Amirkhalili,405
2,3,Kim,Nguyen,380
3,4,John,Smith,80
4,5,George,Brown,260
5,6,Kevin,Miller,135
6,8,Elli,Rahimi,155


In fact, we use subqueries to break more complicated queries into two or more simple ones. We could write this query without a subquery:

In [13]:
q("""
SELECT
    C.CustomerID,
    C.FirstName,
    SUM(SI.SalePrice * SI.Quantity) AS TotalSales
FROM Customer C
JOIN Sale S
  ON C.CustomerID = S.CustomerID
JOIN SaleItem SI
  ON S.SaleID = SI.SaleID
GROUP BY
    C.CustomerID,
    C.FirstName
""")

,CustomerID,FirstName,TotalSales
0,1,Sara,2265
1,2,Amin,405
2,3,Kim,380
3,4,John,80
4,5,George,260
5,6,Kevin,135
6,8,Elli,155


### All subqueries in one view

| Subquery Location | Subquery Output | How Many Times Executed? | Correlated? | Main Purpose |
|---|---|---|---|---|
| SELECT | A single value (scalar) | Once per row | Usually yes | Calculate a column value for each row |
| FROM | A complete table (derived table) | Once | Usually no | Create a temporary table |
| WHERE | A value / list / condition | Depends on the type | Can be either | Filter rows |

## Section 2 — VIEW

**What is a view in SQL?** A view, in fact, is a saved query. That is all. A view is not a table or anything else — just a query, so we do not need to write it out every time.

**Why do we need views?** Views are useful for maintaining confidentiality (by restricting access to selected parts of the database) and for simplifying frequently used query types.

In [14]:
q("""
DROP VIEW IF EXISTS BasicEmployee;
CREATE VIEW BasicEmployee AS
SELECT EmployeeID, FirstName, LastName, ManagerID
FROM Employee
""")

OK — DROP executed.


In [15]:
q("""
SELECT * FROM BasicEmployee
""")

,EmployeeID,FirstName,LastName,ManagerID
0,1,Alan,Stone,NaN
1,2,Ahmad,Karimi,1.0
2,3,Elli,Moore,1.0
3,4,Luke,Perry,2.0
4,5,James,Chen,2.0
5,6,Alex,Diaz,3.0


Sometimes we want to create a view from a **joined** table.

**Query: information about women's shoes made in California.**

⚠️ Attention: like creating tables, before we create a view, make sure there is not a view with that name already.

*(The textbook uses the schema prefix `redcat.Product`; SQLite has no schema prefixes, so we drop it.)*

In [16]:
q("""
DROP VIEW IF EXISTS CalifWomenShoes;
CREATE VIEW CalifWomenShoes AS
SELECT ProductName,
       ManufacturerName,
       Category,
       Color,
       ListPrice
FROM Product P
JOIN Manufacturer M
  ON P.ManufacturerID = M.ManufacturerID
WHERE Gender = 'F'
  AND State = 'CA'
""")

OK — DROP executed.


In [17]:
q("""
SELECT * FROM CalifWomenShoes
""")

,ProductName,ManufacturerName,Category,Color,ListPrice
0,RetroRunner,Fila,Sneakers,Black,85
1,GlamHeel,Fashion Lab,Heels,Black,130
2,CityFlat,Fashion Lab,Flats,Blue,70


In [18]:
q("""
SELECT DISTINCT Color
FROM CalifWomenShoes
WHERE ManufacturerName = 'Fashion Lab'
""")

,Color
0,Black
1,Blue


> ### 💡 Remember
> - A view is saved in the database, so it exists until you delete it: `DROP VIEW ...`
> - A view is **dynamic**: if the base table info changes, the view also changes.

### Views and the outer-join problem (from Notebook 3.3)

Using a view, the WHERE happens **first**; then we join the tables:

In [19]:
q("""
DROP VIEW IF EXISTS DecPurchase;
CREATE VIEW DecPurchase AS
SELECT * FROM Purchase WHERE MONTH(PurchaseDate) = 12
""")

OK — DROP executed.


In [20]:
q("""
SELECT E.EmployeeID, E.FirstName, E.LastName, PurchaseDate, ManufacturerID
FROM Employee E LEFT OUTER JOIN DecPurchase P
  ON E.EmployeeID = P.EmployeeID
""")

,EmployeeID,FirstName,LastName,PurchaseDate,ManufacturerID
0,1,Alan,Stone,None,NaN
1,2,Ahmad,Karimi,2025-12-05,1.0
2,3,Elli,Moore,None,NaN
3,4,Luke,Perry,2025-12-12,2.0
4,5,James,Chen,2025-12-27,3.0
5,6,Alex,Diaz,None,NaN


### Using a view instead of a subquery

Example with a subquery:

In [21]:
q("""
SELECT *
FROM
(
    SELECT Country,
           AVG(Population) AS AvgPopulation
    FROM Population
    GROUP BY Country
) AS P
WHERE AvgPopulation > 2000000
""")

,Country,AvgPopulation
0,Germany,2558167.0
1,USA,5149775.0


Now we use a view to replace the subquery:

In [22]:
q("""
DROP VIEW IF EXISTS CountryAverage;
CREATE VIEW CountryAverage AS
SELECT Country,
       AVG(Population) AS AvgPopulation
FROM Population
GROUP BY Country
""")

OK — DROP executed.


In [23]:
q("""
SELECT * FROM CountryAverage
WHERE AvgPopulation > 2000000
""")

,Country,AvgPopulation
0,Germany,2558167.0
1,USA,5149775.0


So, what is the difference between the subquery and the view here? The subquery runs only one time. But you can use the view as many times as you want in other scripts.

## Section 3 — Temp Tables

Temp tables are temporary tables that can be used to avoid working on base tables — when we just want a part of a table, or we need to replace a query (replacing subqueries), such as a join or a subquery.

*(SQL Server marks temp tables with `#`, e.g. `#temp_amin`. SQLite instead uses `CREATE TEMP TABLE` — same idea: the table lives in memory and disappears when you disconnect.)*

In [24]:
q("""
DROP TABLE IF EXISTS temp_amin;
CREATE TEMP TABLE temp_amin (
    ID INT, FirstName VARCHAR(100), LastName VARCHAR(100), Age INT);
INSERT INTO temp_amin SELECT ID, FirstName, LastName, Age FROM Amin
""")

OK — DROP executed.


In [25]:
q("""
SELECT * FROM temp_amin
""")

,ID,FirstName,LastName,Age
0,1,Amin,Amirkhalili,30
1,2,Amin,Amirkhalili,50
2,3,Sara,Lopez,27
3,4,Elli,Rahimi,35
4,5,Sadra,Kazemi,22
5,6,Hossein,Nouri,44
6,7,Kim,Nguyen,31


As temp tables are built like base tables, we can also build them by copying from a table.

*(SQL Server: `SELECT * INTO #temp_amin FROM Amin`. SQLite:)*

In [26]:
q("""
DROP TABLE IF EXISTS temp_amin2;
CREATE TEMP TABLE temp_amin2 AS
SELECT * FROM Amin
""")

OK — DROP executed.


In [27]:
q("""
SELECT * FROM temp_amin2
""")

,ID,FirstName,LastName,Age,Gender
0,1,Amin,Amirkhalili,30,Male
1,2,Amin,Amirkhalili,50,Male
2,3,Sara,Lopez,27,Female
3,4,Elli,Rahimi,35,Female
4,5,Sadra,Kazemi,22,Male
5,6,Hossein,Nouri,44,Male
6,7,Kim,Nguyen,31,Female


**Using temp tables instead of subqueries:**

In [28]:
q("""
DROP TABLE IF EXISTS CountryAverageTmp;
CREATE TEMP TABLE CountryAverageTmp AS
SELECT Country,
       AVG(Population) AS AvgPopulation
FROM Population
GROUP BY Country
""")

OK — DROP executed.


In [29]:
q("""
SELECT * FROM CountryAverageTmp
WHERE AvgPopulation > 2000000
""")

,Country,AvgPopulation
0,Germany,2558167.0
1,USA,5149775.0


Again, the subquery only runs once. But the temp table stays there, and you can change it without changing any base tables.

The difference between a view and a temp table: if the base table changes, the **view** is updated but the **temp table** is not.

## Section 4 — CTE (Common Table Expression)

A CTE is also used to simplify big queries, helping to replace subqueries.

```sql
WITH CTE AS (SELECT ...)
SELECT * FROM CTE
```

Example (subquery):

In [30]:
q("""
SELECT * FROM
(SELECT * FROM Amin WHERE Gender LIKE 'Male') AS t
""")

,ID,FirstName,LastName,Age,Gender
0,1,Amin,Amirkhalili,30,Male
1,2,Amin,Amirkhalili,50,Male
2,5,Sadra,Kazemi,22,Male
3,6,Hossein,Nouri,44,Male


Now we use a CTE:

In [31]:
q("""
WITH cte_gender AS (SELECT * FROM Amin WHERE Gender LIKE 'Male')
SELECT * FROM cte_gender
""")

,ID,FirstName,LastName,Age,Gender
0,1,Amin,Amirkhalili,30,Male
1,2,Amin,Amirkhalili,50,Male
2,5,Sadra,Kazemi,22,Male
3,6,Hossein,Nouri,44,Male


> ### 💡 Remember
> A CTE only exists if you run the query starting from `WITH ...` every time. If you only run `SELECT * FROM cte_gender` on its own, it gives an error. Try it!

### So, how do we know when to use a subquery, a view, a temp table, or a CTE?

Let's see it metaphorically:

- You are writing a report. In the middle of the report, you need to calculate the average 2026 sale amount of all products. This is a one-time job. **You use a subquery. DONE.**
- Now, imagine that before you start the report, you calculate all the data you need up front, then use it in your report. **You use a CTE.**
- Now, imagine you calculate something and write it on a piece of paper. You use it while writing the report, then throw it away. **You use a temp table.** You work on it — add something, update, delete, ...
- Now imagine you create a notebook, and later you can always refer back to it. **You have a view.**

### Golden Comparison Table

| Feature | Subquery | CTE | Temp Table | View |
|---|---|---|---|---|
| What is it? | A query inside another query | A named query | A temporary physical table | A saved query |
| Stores data? | ❌ No | ❌ No | ✅ Yes | ❌ No (usually) |
| Lifetime | Only during that part of the query | Only during the current query | Until the session ends | Until it is dropped |
| Reusable? | ❌ No | Only within the same query | ✅ Yes, within the same session | ✅ Yes, permanently |
| Readability | Low | High | High | High |
| Good for multi-step processing? | ❌ No | Limited | ✅ Yes | ❌ No |

### 5-Second Rule (For Exams)

| If the question asks for... | Use |
|---|---|
| A query inside another query | **Subquery** |
| Making a large query more readable | **CTE** |
| Reusing the same query throughout the project | **View** |
| Storing intermediate results for multi-step processing | **Temp Table** |

## Section 5 — Using PARTITION BY + CTE

Suppose we have countries with COVID records over time. We want to calculate a measure based on their **newest** data. First, we create a CTE with ROW_NUMBER():

In [32]:
q("""
WITH Latest AS (
    SELECT
        location,
        continent,
        date,
        total_cases,
        population,
        ROW_NUMBER() OVER (PARTITION BY location ORDER BY date DESC) AS rn
    FROM CovidDeaths
)
SELECT * FROM Latest
""")

,location,continent,date,total_cases,population,rn
0,Canada,North America,2021-07-01,1412000,38000000,1
1,Canada,North America,2021-06-15,1401000,38000000,2
2,Canada,North America,2021-06-01,1385000,38000000,3
3,Germany,Europe,2021-07-01,3730000,83000000,1
4,Germany,Europe,2021-06-15,3717000,83000000,2
5,Germany,Europe,2021-06-01,3700000,83000000,3
6,Iran,Asia,2021-07-01,3230000,85000000,1
7,Iran,Asia,2021-06-15,3020000,85000000,2
8,Iran,Asia,2021-06-01,2950000,85000000,3


What `ROW_NUMBER() OVER (PARTITION BY location ORDER BY date DESC)` does is divide the table into sub-tables based on countries, count and name the rows as `rn`, and order them from the newest date to the oldest.

Then we want to select only the newest data, which means `rn = 1`:

*(SQL Server uses `CAST(... AS DECIMAL(16,8))`; in SQLite we cast to REAL.)*

In [33]:
q("""
WITH Latest AS (
    SELECT
        location,
        continent,
        date,
        total_cases,
        population,
        ROW_NUMBER() OVER (PARTITION BY location ORDER BY date DESC) AS rn
    FROM CovidDeaths
)
SELECT
    location,
    date,
    (CAST(total_cases AS REAL) / NULLIF(population, 0)) * 100 AS LatestInfectionRate
FROM Latest
WHERE continent IS NOT NULL
  AND rn = 1
ORDER BY LatestInfectionRate DESC
""")

,location,date,LatestInfectionRate
0,Germany,2021-07-01,4.493976
1,Iran,2021-07-01,3.800000
2,Canada,2021-07-01,3.715789


---
### 🎯 Your turn — practice ideas

1. Using a subquery, find the products that have **never** been sold.
2. Create a view `BostonCustomers` and query it.
3. Use a CTE + ROW_NUMBER() to find each customer's **most recent** sale.

---
⏭️ **Coming up in Notebook 3.5:** Data Cleaning with SQL.